In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    root_mean_squared_error,
    mean_squared_error
)

In [2]:
df = pd.read_csv('src/data/training/processed/final_training_data.csv')

In [3]:
target_cols = df.filter(regex='^(dn__|ul__|mv__|madrid__|ep__|cb__|ob__|ab__)').columns

missing_or_zero = df[target_cols].isna() | (df[target_cols] == 0)

missing_fraction = missing_or_zero.mean(axis=1)

df = df[missing_fraction <= 0.8].copy()

In [4]:
y = df[['t__ccf']]
X = df[['ob__avg_util_ref']]

In [5]:
bins = [0, 0.2, 0.4, 0.6, 0.8]
labels = ['<=20%', '20% to 40%', '40% to 60%', '60% to 80%']

df['util_bin'] = pd.cut(df['ob__avg_util_ref'], bins=bins, labels=labels, include_lowest=True)

dummies = pd.get_dummies(df['util_bin'])

X = pd.concat([df[['ob__avg_util_ref']], dummies], axis=1)
X = X.drop(columns=['ob__avg_util_ref'])
X = X.astype(int)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [7]:
lin_reg = LinearRegression().fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

In [8]:
import statsmodels.api as sm
X_train_sm = sm.add_constant(X_train)

model = sm.OLS(y_train, X_train_sm).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 t__ccf   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     25.27
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           7.28e-21
Time:                        04:43:13   Log-Likelihood:                -6573.3
No. Observations:               10380   AIC:                         1.316e+04
Df Residuals:                   10375   BIC:                         1.319e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6343      0.007     97.204      0.0

In [9]:
X_test_sm = sm.add_constant(X_test)
y_pred = model.predict(X_test_sm)

In [10]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print("MSE:", mse, "RMSE:", rmse)

MSE: 0.20880208336744974 RMSE: 0.4569486660090494
